# PydanticAI Agent

Replaces the OpenAI SDK + toyaikit setup with PydanticAI — a higher-level framework that handles tool registration, the agentic loop, and message history for you.

## Concepts covered
- **`Agent`**: PydanticAI's agent wraps the LLM, tools, and instructions in one object
- **Tools passed as methods**: pass `instance.method` directly — PydanticAI reads the docstring
- **`agent.run()`**: async version of the agentic loop (use `run_sync` in scripts/notebooks)
- **`message_history`**: pass prior messages to maintain multi-turn conversation
- **Message inspection**: understanding the structure of `result.all_messages()` — requests, responses, tool calls, tool returns

In [ ]:
# !uv pip install "pydantic-ai>=0.1.1"

In [1]:
instructions = """
You're a documentation assistant.

Answer the user question using only the documentation knowledge base.

Make 3 iterations:

1) First iteration:
   - Perform one search using the search tool to identify potentially relevant documents.
   - Explain (in 2–3 sentences) why this search query is appropriate for the user question.

2) Second iteration:
   - Analyze the results from the previous search.
   - Based on the filenames or documents returned, perform:
       - Up to 2 additional search queries to refine or expand coverage, and
       - One or more get_file calls to retrieve the full content of the most relevant documents.
   - For each search or get_file call, explain (in 2–3 sentences) why this action is necessary and how it helps answer the question.

3) Third iteration:
   - Analyze the retrieved document contents from get_file.
   - Synthesize the information from these documents into a final answer to the user.

IMPORTANT:
- At every step, explicitly explain your reasoning for each search query or file retrieval.
- Use only facts found in the documentation knowledge base.
- Do not introduce outside knowledge or assumptions.
- If the answer cannot be found in the retrieved documents, clearly inform the user.

Additional notes:
- The knowledge base is entirely about Evidently, so you do not need to include the word "evidently" in search queries.
- Prefer retrieving and analyzing full documents (via get_file) before producing the final answer.
"""

In [6]:
# index, highlighter, fileindex

from gitsource import GithubRepositoryDataReader
from minsearch import AppendableIndex

from minsearch import Highlighter, Tokenizer
from minsearch.tokenizer import DEFAULT_ENGLISH_STOP_WORDS

reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)
files = reader.read()

parsed_docs = [doc.parse() for doc in files]

index = AppendableIndex(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)
index.fit(parsed_docs)


stopwords = DEFAULT_ENGLISH_STOP_WORDS | {"evidently"}
highlighter = Highlighter(
    highlight_fields=["content"],
    max_matches=3,
    snippet_size=50,
    tokenizer=Tokenizer(stemmer="snowball", stop_words=stopwords)
)

file_index = {doc['filename']: doc['content'] for doc in parsed_docs}

## Setup: Index + Highlighter + File Index

Three components:
- **index**: keyword search over full (unchunked) documents
- **highlighter**: extracts relevant snippets from results so the LLM doesn't get the full 10k-char doc
- **file_index**: a plain dict mapping filename → full content, used by `get_file`

In [7]:
from typing import Any, Dict, List


class SearchTools:
    """
    Provides search and file retrieval utilities over an indexed data store.
    """

    def __init__(
        self,
        index: Any,
        highlighter: Any,
        file_index: Dict[str, str]
    ) -> None:
        """
        Initialize the SearchTools instance.

        Args:
            index: Searchable index providing a `search` method.
            highlighter: Highlighter used to annotate search results.
            file_index (Dict[str, str]): Mapping of filenames to file contents.
        """
        self.index = index
        self.highlighter = highlighter
        self.file_index = file_index

    def search(self, query: str) -> List[Dict[str, Any]]:
        """
        Search the index for results matching a query and highlight them.

        Args:
            query (str): The search query to look up in the index.

        Returns:
            List[Dict[str, Any]]: A list of highlighted search result objects.
        """
        search_results = self.index.search(
            query=query,
            num_results=5
        )

        return self.highlighter.highlight(query, search_results)

    def get_file(self, filename: str) -> str:
        """
        Retrieve a file's contents by filename.

        Args:
            filename (str): The filename of the file to retrieve.

        Returns:
            str: The file contents if found, otherwise an error message.
        """
        if filename in self.file_index:
            return self.file_index[filename]
        return f"file {filename} does not exist"

In [9]:
search_tools = SearchTools(
                index=index, 
                highlighter=highlighter, 
                file_index=file_index)

tools = [search_tools.search, search_tools.get_file]

In [25]:
from pydantic_ai import Agent, RunUsage

agent = Agent(
    name='search',
    model='openai:gpt-4o-mini',
    instructions=instructions,
    tools=tools
)

In [13]:
query = 'how do I use evidently to monitor my machine learnign models?'

In [ ]:
# agent.run() is async — use `await` in Jupyter notebooks.
# In a regular Python script, use agent.run_sync() instead.
result = await agent.run(query)

In [19]:
print(result.output)

The retrieved documents provide essential insights on how to monitor machine learning models using Evidently.

1. The "Overview" document explains different methods for setting up monitoring, including batch monitoring jobs and tracing with scheduled evaluations. It outlines how to create evaluation pipelines, run metric calculations, and visualize results. This is crucial for the user to set up an effective monitoring system for their ML models, as it covers practical implementation steps and benefits of each approach.

2. The introduction document outlines what Evidently is, including features available both in the open-source library and in the cloud platform. It highlights the tools available for evaluating model performance and data quality, which directly relates to monitoring machine learning models.

With this information synthesized, here's a more finalized answer to the user's question on how to use Evidently to monitor their machine learning models.

### Final Answer:
To use

In [20]:
messages = result.all_messages()

In [21]:
result.usage()

RunUsage(input_tokens=3969, cache_read_tokens=1152, output_tokens=577, details={'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, requests=3, tool_calls=3)

In [ ]:
# Inspect the full message history to see what happened inside the agent loop.
# Each message has a `kind` (request/response) and `parts` with different part_kinds:
#   - 'user-prompt': the user's message
#   - 'tool-call': the LLM requesting a tool (name + args)
#   - 'tool-return': the result we sent back after executing the tool
#   - 'text': the LLM's final text answer

for m in messages:
    print(m.kind)
    for p in m.parts:
        part_kind = p.part_kind
        if part_kind == 'user-prompt':
            print('USER:', p.content)
        if part_kind == 'tool-call':
            print('TOOL CALL:', p.tool_name, p.args)
        if part_kind == 'tool-return':
            print('TOOL RETURN:', p.tool_name)
        if part_kind == 'text':
            print(p.content)
    print()

In [32]:
def print_messages(messages):
    for m in messages:
        print(m.kind)
        for p in m.parts:
            part_kind = p.part_kind
            if part_kind == 'user-prompt':
                print('USER:', p.content)
            if part_kind == 'tool-call':
                print('TOOL CALL:', p.tool_name, p.args)
            if part_kind == 'tool-return':
                print('TOOL RETURN:', p.tool_name)
            if part_kind == 'text':
                print(p.content)
        print()

In [ ]:
messages = []
usage = RunUsage()

# Q&A loop: each turn passes the full message history so the agent remembers context.
# result.new_messages() returns only the messages from this turn (not the full history),
# which we append to carry context forward into the next turn.
while True:
    user_prompt = input('You:')
    if user_prompt.lower().strip() == 'stop':
        break

    result = await agent.run(user_prompt, message_history=messages)
    usage = usage + result.usage()

    print_messages(result.new_messages())
    messages.extend(result.new_messages())